# Afρ photometry — validation notebook

Validates the photometry half of `ztfcomet` on a single test target
(**24P/Schaumasse**), stage by stage. Run `query.ipynb` first so there are
cutouts on disk.

This notebook also **demonstrates the four corrections** that separate
`ztfcomet.phot` from the old `afrho_240P.ipynb`, with the difference measured on
real frames rather than asserted:

| | correction |
|---|---|
| C1 | per-frame zeropoint (the old code broadcast one frame's `MAGZP` over the whole table) |
| C4 | ZTF colour term `CLRCOEFF · (g−r)` |
| C5 | aperture correction from `APCOR1..6` |
| C8 | uncertainties propagated all the way to `afrho0_cm_err` |

Frames failing a quality check are **flagged, never dropped**.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

# Make the package importable from a checkout, wherever the kernel started.
_root = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
              if (p / "ztfcomet" / "__init__.py").is_file()), None)
if _root and str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import astropy.units as u
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import ztfcomet as zc

zc.setup_logging()

target  = zc.get_target("24P")
DATADIR = zc.data_dir(target.name)
FIGDIR  = zc.fig_dir(target.name)
print(DATADIR, "\n")
print(target.phot)

## 1. Frame inventory

`build_frame_table` replaces `ccdproc.ImageFileCollection`, which silently
skipped files it could not open — so the 245-byte HTML "FITS" files from a bad
download simply vanished from the analysis with no message. Unreadable files are
reported here.

In [ ]:
table = zc.build_frame_table(DATADIR)
table.insert(1, "target", target.name)

print(f"{len(table)} frames")
table[["file", "isot", "filter", "exptime", "egain", "readnoise",
       "pixscale", "fwhm_pix", "zpmag", "clrcoeff", "maglim"]].head()

In [ ]:
# The header keywords the corrections depend on.
print(f"MAGZP    : {table.zpmag.min():.3f} .. {table.zpmag.max():.3f}"
      f"   (spread {table.zpmag.max() - table.zpmag.min():.2f} mag)")
print(f"CLRCOEFF : {table.clrcoeff.min():.4f} .. {table.clrcoeff.max():.4f}")
print(f"MAGZPRMS : median {table.zpmagrms.median():.4f} mag")
print(f"SEEING   : {table.fwhm_pix.min():.2f} .. {table.fwhm_pix.max():.2f} px")
print()
print("A single zeropoint for all frames would therefore be wrong by up to "
      f"{table.zpmag.max() - table.zpmag.min():.2f} mag "
      f"= a factor of {10 ** (0.4 * (table.zpmag.max() - table.zpmag.min())):.1f} in flux.")

## 2. Ephemerides

Joined **on JD**, and the orbit record is chosen per epoch via
`Target.resolve_orbit_record` — so a comet like 240P/NEAT, whose solution
changes between apparitions, needs no per-notebook special-casing.

In [ ]:
table = zc.attach_ephemerides(table, target)
table[["isot", "orb_id", "ra", "dec", "r", "delta", "alpha", "elong",
       "sunTargetPA", "velocityPA"]].head()

## 3. Aperture photometry

Aperture radius is the physical `rho_km` at the comet, so it shrinks in pixels
as Δ grows. `rho_fwhm` records whether the aperture still contains the PSF —
the old notebook computed exactly this diagnostic and then never used it.

In [ ]:
table = zc.measure_photometry(table, DATADIR, target.phot)

print(f"rho_pix  : {table.rho_pix.min():.1f} .. {table.rho_pix.max():.1f} px")
print(f"rho/FWHM : {table.rho_fwhm.min():.2f} .. {table.rho_fwhm.max():.2f}"
      f"   (threshold {target.phot.min_rho_fwhm})")
print(f"SNR      : {table.snr.min():.1f} .. {table.snr.max():.1f}")

table[["file", "x_center", "y_center", "centroid_shift_pix", "edge_dist_pix",
       "rho_pix", "source_sum", "source_sum_err", "snr", "inst_mag"]].head()

### Quality flags

Nothing is dropped. Each flag says why a measurement should not be trusted, and
the row keeps its numbers so you can look at it.

In [ ]:
flag_counts = {c.replace("flag_", ""): int(table[c].sum())
               for c in zc.FLAG_COLUMNS if table[c].any()}
print(f"flagged frames: {int((table[zc.FLAG_COLUMNS].any(axis=1)).sum())} / {len(table)}")
for name, n in sorted(flag_counts.items(), key=lambda kv: -kv[1]):
    print(f"  {name:18s} {n}")

In [ ]:
# Look at a flagged frame rather than trusting the flag blindly.
flagged = table[table[zc.FLAG_COLUMNS].any(axis=1)]
if len(flagged):
    row = flagged.iloc[0]
    print(row.file, "->", [c.replace("flag_", "") for c in zc.FLAG_COLUMNS if row[c]])
    zc.plot_cutout(DATADIR / row.file, row=row, target=target)
    plt.show()
else:
    print("no flagged frames in this set")

## 3b. Background-source contamination

A comet drifts across the star field, so on some frames a background star lands
inside the aperture. The extra flux is indistinguishable from cometary activity
in the image alone and reads out as a spurious Afρ spike.

`flag_contamination` sums the Gaia DR3 flux inside the aperture into one
effective magnitude

$$G_\mathrm{eff} = -2.5\log_{10}\sum_i 10^{-0.4G_i}$$

(two $G=15$ stars give $G_\mathrm{eff}=14.25$) and compares it with the comet's
predicted `Tmag` from JPL. The frame is flagged when the background carries
≥30% of the comet's flux — equivalently $G_\mathrm{eff} \le T_\mathrm{mag}+1.31$.

The search radius is the aperture **plus one seeing FWHM**, since a star just
outside still spills flux in through the PSF wings.

In [ ]:
cat = zc.GaiaCatalog()
print(cat.describe())

# The worked example from the specification.
print(f"\nG_eff of two G=15 sources : {zc.effective_magnitude([15.0, 15.0]):.4f}")
print(f"30% threshold in magnitudes : Tmag + {-2.5 * np.log10(0.30):.4f}")

In [ ]:
table = zc.flag_contamination(table, target.phot)

n = int(table.flag_contaminated.sum())
print(f"contaminated frames: {n} / {len(table)}")

cols = ["isot", "filter", "tmag", "contam_radius_arcsec", "contam_n_sources",
        "contam_g_eff", "contam_g_brightest", "contam_ratio", "flag_contaminated"]
table[table.flag_contaminated][cols] if n else table[cols].head()

### Does the flag actually catch the outliers?

The honest check is not whether the flag fires, but whether it fires on the
frames that were anomalous for an independent reason.

In [ ]:
if int(table.flag_contaminated.sum()):
    # Afrho needs calibration first; do a provisional pass just for this check.
    prov = zc.compute_afrho(zc.calibrate(table, target.phot), target.phot)
    clean = prov[~prov.flag_contaminated]
    dirty = prov[prov.flag_contaminated]

    print(f"  clean frames       : A(0)frho {clean.afrho0_cm.min():.0f}-{clean.afrho0_cm.max():.0f} cm"
          f"  (median {clean.afrho0_cm.median():.0f})")
    print(f"  contaminated frames: A(0)frho {dirty.afrho0_cm.min():.0f}-{dirty.afrho0_cm.max():.0f} cm"
          f"  (median {dirty.afrho0_cm.median():.0f})")

    top = prov.nlargest(6, "afrho0_cm")
    print(f"\n  of the 6 brightest frames, {int(top.flag_contaminated.sum())} are contamination-flagged")
else:
    print("no contaminated frames in this set")

In [ ]:
# Look at one, rather than trusting the flag.
hit = table[table.flag_contaminated]
if len(hit):
    row = hit.iloc[0]
    print(f"{row.file}\n  {row.contam_n_sources} Gaia source(s), G_eff={row.contam_g_eff:.2f} "
          f"vs Tmag={row.tmag:.2f}  ->  {row.contam_ratio:.2f}x the comet flux")
    zc.plot_cutout(DATADIR / row.file, row=row, target=target, phot_config=target.phot)
    plt.show()

Two limits to state with any result: the catalogue is complete only to
$G=18.5$, so "uncontaminated" means "no *catalogued* source" — fainter stars and
galaxies are invisible to this test. And Gaia $G$ is compared directly with a
visual `Tmag`; the passbands differ by ~0.1–0.2 mag for typical stellar colours,
small against a 0.28 mag threshold but not zero.

## 4. Calibration — C1, C4, C5

`calibrate` applies, in order:

```
filter_mag = inst_mag + MAGZP + CLRCOEFF·(g−r) + APCOR(ρ)
             ^^^^^^^^^^^^^^^^   ^^^^^^^^^^^^^^   ^^^^^^^^^
             C1: per frame      C4: colour       C5: aperture
```

The colour is measured from same-night g/r pairs where they exist, and falls
back to `PhotConfig.default_color_gr` with `flag_color_default` set.

In [ ]:
table = zc.calibrate(table, target.phot)

print(f"colour measured from g/r pairs : {int((~table.flag_color_default).sum())} / {len(table)}")
print(f"g-r used                       : {table.color_gr.min():.3f} .. {table.color_gr.max():.3f}")
print(f"C4 colour term  |shift|        : median {table.color_term.abs().median():.4f} mag, "
      f"max {table.color_term.abs().max():.4f} mag")
print(f"C5 aperture corr               : median {table.apcor.median():.4f} mag, "
      f"range {table.apcor.min():.4f} .. {table.apcor.max():.4f}")
print(f"C8 mag uncertainty             : median {table.filter_mag_err.median():.4f} mag")

table[["isot", "filter", "inst_mag", "zpmag", "color_gr", "color_term",
       "apcor", "filter_mag", "filter_mag_err"]].head()

### C1 in numbers

Reproduce what the old notebook actually did — `inst_mag + row.zpmag`, where
`row` was the leftover loop variable, i.e. the **last** frame's zeropoint — and
compare.

In [ ]:
old_mag = table.inst_mag + table.zpmag.iloc[-1]     # the bug, exactly
new_mag = table.inst_mag + table.zpmag              # per frame

delta = (new_mag - old_mag).abs()
good  = table[zc.FLAG_COLUMNS].any(axis=1).pipe(lambda s: ~s)

print(f"magnitude error from the single-zeropoint bug:")
print(f"  all frames      : median {delta.median():.3f} mag, max {delta.max():.3f} mag"
      f"  -> up to {10 ** (0.4 * delta.max()):.1f}x in flux")
print(f"  unflagged only  : median {delta[good].median():.3f} mag, max {delta[good].max():.3f} mag"
      f"  -> up to {10 ** (0.4 * delta[good].max()):.1f}x in flux")
print(f"  frames off >10% in Afrho: {int((delta[good] > 0.105).sum())} / {int(good.sum())}")
print()
print("This is epoch-dependent scatter, not a constant offset, so it distorts")
print("lightcurve SHAPE -- the thing an Afrho study is trying to measure.")

## 5. Afρ — C8

$$Af\rho = \frac{4\,\Delta^2 r_h^2}{\rho}\,10^{-0.4(m_c - m_\odot)},
\qquad A(0°)f\rho = Af\rho \cdot 10^{0.4\beta\alpha}$$

Uncertainties propagate photon and sky noise, `MAGZPRMS`, the colour-term
uncertainty and its covariance with the zeropoint (`ZPCLRCOV`).

In [ ]:
table = zc.compute_afrho(table, target.phot)

good = table[table.quality_ok]
print(f"unflagged frames : {len(good)} / {len(table)}")
print(f"A(0)frho         : {good.afrho0_cm.min():.1f} .. {good.afrho0_cm.max():.1f} cm"
      f"   (rho = {target.phot.rho_km:.0f} km)")
print(f"fractional error : median {(good.afrho0_cm_err / good.afrho0_cm).median():.3f}")

table[["isot", "filter", "filter_mag", "filter_mag_err", "r", "delta", "alpha",
       "afrho_cm", "afrho_cm_err", "phase_corr", "afrho0_cm", "afrho0_cm_err",
       "quality_ok"]].head(10)

### Unit check

Worth doing once by hand: Δ carries AU, ρ carries km, and $r_h$ enters as a bare
number in AU. Astropy resolves AU²/km to a length, and `.to_value(u.cm)` gives cm.

In [ ]:
row = good.iloc[0]
manual = (4 * (row.delta * u.au) ** 2 * row.r ** 2 / (row.rho_km * u.km)
          * 10 ** (-0.4 * (row.filter_mag - row.solmag))).to_value(u.cm)

print(f"module : {row.afrho_cm:.4f} cm")
print(f"by hand: {manual:.4f} cm")
assert np.isclose(manual, row.afrho_cm, rtol=1e-9)
print("units check out")

## 6. Lightcurve and save

In [ ]:
ax = zc.plot_afrho({target.name: table},
                   filters=[f for f in ("ZTF_g", "ZTF_r") if (table["filter"] == f).any()],
                   x="rh")
ax.set_title(f"{target.name}   " + r"$\rho$ = " + f"{target.phot.rho_km:.0f} km")
plt.show()

In [ ]:
outpath = zc.result_dir(target.name) / f"photometry_{zc.target_slug(target.name)}.csv"
table.to_csv(outpath, index=False)
print(f"wrote {outpath}  ({len(table)} rows, {len(table.columns)} columns)")

The whole chain above is `zc.run_photometry(target)`, which is what
`notebooks/main.py` calls. This notebook exists to check the stages; use the
script for real runs.

In [ ]:
# Equivalent one-liner:
# table = zc.run_photometry(target)